# Явная разностная схема для уравнения теплопроводности

**Лаборатория математики ФН1** — демонстрация к докладу на научном семинаре

Рассматривается первая краевая задача

$$\frac{\partial u}{\partial t} = a^2\frac{\partial^2 u}{\partial x^2},\quad 0<x<L,\ t>0,$$
$$u(x,0)=\varphi(x),\qquad u(0,t)=u(L,t)=0.$$

Цель — показать, как условие устойчивости $\sigma = a^2\tau/h^2 \le 1/2$
проявляется в расчёте, и измерить наблюдаемый порядок точности.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 12, "grid.alpha": 0.3})

A = 1.0    # коэффициент температуропроводности
L = 1.0    # длина отрезка
T = 0.05   # время расчёта

## Точное решение для проверки

При $\varphi(x)=\sin(\pi x/L)$ решение имеет вид
$u(x,t)=\sin(\pi x/L)\,e^{-a^2\pi^2 t/L^2}$.

In [ ]:
def exact(x, t, a=A, length=L):
    return np.sin(np.pi * x / length) * np.exp(-(a**2) * np.pi**2 * t / length**2)


def solve_explicit(n_x, sigma, a=A, length=L, t_end=T):
    """Явная схема. sigma = a^2 * tau / h^2 задаёт шаг по времени."""
    h = length / (n_x - 1)
    tau = sigma * h**2 / a**2
    n_t = int(round(t_end / tau))

    x = np.linspace(0, length, n_x)
    u = np.sin(np.pi * x / length)

    for _ in range(n_t):
        u[1:-1] = u[1:-1] + sigma * (u[2:] - 2 * u[1:-1] + u[:-2])
        u[0] = u[-1] = 0.0

    return x, u, n_t * tau

## 1. Устойчивый и неустойчивый расчёт

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, sigma in zip(axes, (0.4, 0.52)):
    x, u, t_reached = solve_explicit(41, sigma)
    ax.plot(x, exact(x, t_reached), "k--", lw=2, label="точное решение")
    ax.plot(x, u, lw=1.8, label="разностная схема")
    ax.set_title(f"sigma = {sigma}" + ("  (устойчиво)" if sigma <= 0.5 else "  (неустойчиво)"))
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

## 2. Наблюдаемый порядок точности

При фиксированном $\sigma$ схема имеет порядок $O(h^2)$, поэтому при
измельчении сетки вдвое погрешность должна падать примерно вчетверо.

In [ ]:
print(f"{'узлов':>7} {'h':>10} {'max ошибка':>14} {'порядок':>10}")
previous = None
for n_x in (11, 21, 41, 81, 161):
    x, u, t_reached = solve_explicit(n_x, sigma=0.4)
    error = np.max(np.abs(u - exact(x, t_reached)))
    order = "" if previous is None else f"{np.log2(previous / error):.2f}"
    print(f"{n_x:>7} {L / (n_x - 1):>10.5f} {error:>14.3e} {order:>10}")
    previous = error

## Вопросы к обсуждению

1. Как изменится картина при переходе к неявной схеме и почему она безусловно устойчива?
2. Что произойдёт с наблюдаемым порядком, если фиксировать не $\sigma$, а шаг $\tau$?
3. Как обобщить расчёт на случай переменного коэффициента $a(x)$?